In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from filterpy.kalman import KalmanFilter
import statsmodels.api as sm
import logging
from datetime import datetime, timedelta

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler('stock_selection.log'), logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# Hardcoded Nifty 50 tickers (as of 2023; ideally fetch dynamically in production)
NIFTY50 = [
    'RELIANCE.NS', 'HDFCBANK.NS', 'INFY.NS', 'HDFC.NS', 'ICICIBANK.NS',
    'TCS.NS', 'KOTAKBANK.NS', 'HINDUNILVR.NS', 'ITC.NS', 'AXISBANK.NS',
    'SBIN.NS', 'LT.NS', 'BAJFINANCE.NS', 'ASIANPAINT.NS', 'MARUTI.NS',
    'BHARTIARTL.NS', 'HCLTECH.NS', 'SUNPHARMA.NS', 'TITAN.NS', 'TECHM.NS',
    'NESTLEIND.NS', 'ULTRACEMCO.NS', 'POWERGRID.NS', 'NTPC.NS', 'BAJAJFINSV.NS',
    'INDUSINDBK.NS', 'M&M.NS', 'HDFCLIFE.NS', 'ADANIPORTS.NS', 'JSWSTEEL.NS',
    'TATAMOTORS.NS', 'GRASIM.NS', 'DIVISLAB.NS', 'HEROMOTOCO.NS', 'SHREECEM.NS',
    'BPCL.NS', 'EICHERMOT.NS', 'BRITANNIA.NS', 'CIPLA.NS', 'SBILIFE.NS',
    'WIPRO.NS', 'ONGC.NS', 'COALINDIA.NS', 'UPL.NS', 'DRREDDY.NS',
    'TATASTEEL.NS', 'IOC.NS', 'GAIL.NS', 'BAJAJ-AUTO.NS', 'APOLLOHOSP.NS'
]

# Parameters (tunable via training set)
KALMAN_PROCESS_VARIANCE = 1e-5
KALMAN_MEASUREMENT_VARIANCE = 1.0
REGRESSION_WINDOW = 20
MIN_DATA_POINTS = 100
SIGNIFICANCE_LEVEL = 0.05
TRAIN_TEST_SPLIT = 0.8
TRANSACTION_COST = 0.001  # 0.1% per trade

def fetch_data(stocks, start_date, end_date):
    """Fetch historical adjusted closing prices."""
    try:
        data = yf.download(stocks, start=start_date, end=end_date, progress=False)['Adj Close']
        if data.empty:
            raise ValueError("No data fetched from yfinance.")
        logger.info(f"Fetched data for {len(stocks)} stocks from {start_date} to {end_date}")
        return data
    except Exception as e:
        logger.error(f"Error fetching data: {e}")
        raise

def preprocess_data(data):
    """Handle missing values and ensure sufficient data."""
    data = data.fillna(method='ffill').dropna(axis=1, how='all')
    valid_stocks = [col for col in data.columns if len(data[col].dropna()) >= MIN_DATA_POINTS]
    if len(valid_stocks) < 5:
        raise ValueError(f"Insufficient valid stocks: {len(valid_stocks)}")
    data = data[valid_stocks]
    logger.info(f"Preprocessed data: {len(valid_stocks)} stocks retained")
    return data

def split_data(data, train_test_split=TRAIN_TEST_SPLIT):
    """Split data into training and testing sets."""
    train_size = int(len(data) * train_test_split)
    train_data = data.iloc[:train_size]
    test_data = data.iloc[train_size:]
    logger.info(f"Data split: Training {train_data.index[0]} to {train_data.index[-1]}, Testing {test_data.index[0]} to {test_data.index[-1]}")
    return train_data, test_data

def apply_kalman_filter(series, process_variance=KALMAN_PROCESS_VARIANCE, measurement_variance=KALMAN_MEASUREMENT_VARIANCE):
    """Apply Kalman filter to smooth the series."""
    try:
        kf = KalmanFilter(dim_x=2, dim_z=1)
        kf.x = np.array([series.iloc[0], 0])  # [position, velocity]
        kf.P *= 1e-2  # Initial uncertainty
        kf.R = measurement_variance  # Measurement noise
        kf.Q = np.array([[process_variance, 0], [0, process_variance]])  # Process noise
        kf.F = np.array([[1, 1], [0, 1]])  # State transition
        kf.H = np.array([[1, 0]])  # Observation model
        filtered = []
        for measurement in series:
            kf.predict()
            kf.update(np.array([measurement]))
            if np.isnan(kf.x[0]):
                raise ValueError("Kalman filter produced NaN values")
            filtered.append(kf.x[0])
        return pd.Series(filtered, index=series.index)
    except Exception as e:
        logger.error(f"Kalman filter error on {series.name}: {e}")
        return pd.Series(np.nan, index=series.index)

def rolling_regression(series, window=REGRESSION_WINDOW):
    """Perform rolling linear regression and return slopes, p-values, and t-stats."""
    try:
        slopes, p_values, t_stats = [], [], []
        for i in range(window, len(series)):
            y = series.iloc[i-window:i]
            if y.isna().any():
                slopes.append(np.nan)
                p_values.append(np.nan)
                t_stats.append(np.nan)
                continue
            x = np.arange(window)
            x = sm.add_constant(x)
            model = sm.OLS(y, x).fit()
            slopes.append(model.params[1])
            p_values.append(model.pvalues[1])
            t_stats.append(model.tvalues[1])
        return (pd.Series(slopes, index=series.index[window:]),
                pd.Series(p_values, index=series.index[window:]),
                pd.Series(t_stats, index=series.index[window:]))
    except Exception as e:
        logger.error(f"Rolling regression error on {series.name}: {e}")
        return (pd.Series(np.nan, index=series.index[window:]),
                pd.Series(np.nan, index=series.index[window:]),
                pd.Series(np.nan, index=series.index[window:]))

def calculate_volatility(data, window=REGRESSION_WINDOW):
    """Calculate rolling volatility (std of returns)."""
    returns = data.pct_change().dropna()
    volatility = returns.rolling(window=window).std()
    return volatility

def select_momentum_stocks(slopes_df, p_values_df, t_stats_df, volatility_df, date, n=5):
    """Select top n stocks with strongest significant momentum, normalized by volatility."""
    try:
        slopes = slopes_df.loc[date]
        p_values = p_values_df.loc[date]
        t_stats = t_stats_df.loc[date]
        volatility = volatility_df.loc[date]
        significant = p_values < SIGNIFICANCE_LEVEL
        if significant.sum() == 0:
            logger.warning(f"No significant trends on {date}")
            return []
        significant_slopes = slopes[significant]
        significant_volatility = volatility[significant]
        normalized_slopes = significant_slopes / significant_volatility
        significant_t_stats = t_stats[significant]
        top_n = normalized_slopes.abs().nlargest(n).index
        weights = significant_t_stats[top_n] / significant_t_stats[top_n].sum()
        logger.info(f"Selected stocks on {date}: {top_n.tolist()}")
        return list(zip(top_n, weights))
    except Exception as e:
        logger.error(f"Error selecting stocks on {date}: {e}")
        return []

def evaluate_performance(data, index_data, selected_stocks, selection_dates):
    """Evaluate strategy performance over the test period with transaction costs and risk metrics."""
    returns, turnover = [], []
    old_positions = pd.Series(0, index=data.columns)
    for i in range(len(selection_dates) - 1):
        t = selection_dates[i]
        t_next = selection_dates[i + 1]
        selected = selected_stocks.get(t, [])
        if not selected:
            continue
        try:
            stocks, weights = zip(*selected)
            new_positions = pd.Series(weights, index=stocks).reindex(data.columns, fill_value=0)
            turnover.append(abs(new_positions - old_positions).sum() / 2)
            prices_t = data.loc[t, stocks]
            prices_t_next = data.loc[t_next, stocks]
            stock_returns = (prices_t_next / prices_t) - 1
            portfolio_return = (stock_returns * weights).sum()
            transaction_cost = turnover[-1] * TRANSACTION_COST
            portfolio_return -= transaction_cost
            index_return = (index_data.loc[t_next] / index_data.loc[t]) - 1
            returns.append({
                'date': t,
                'stocks': stocks,
                'weights': weights,
                'portfolio_return': portfolio_return,
                'index_return': index_return
            })
            old_positions = new_positions
        except Exception as e:
            logger.error(f"Error computing returns for {t}: {e}")
    returns_df = pd.DataFrame(returns)
    if not returns_df.empty:
        returns_df['strategy_cum_return'] = (1 + returns_df['portfolio_return']).cumprod() - 1
        returns_df['index_cum_return'] = (1 + returns_df['index_return']).cumprod() - 1
        # Calculate risk metrics
        risk_free_rate = 0.06 / 52  # 6% annual risk-free rate, weekly
        excess_returns = returns_df['portfolio_return'] - risk_free_rate
        sharpe_ratio = excess_returns.mean() / excess_returns.std() * np.sqrt(52)
        cumulative_returns = (1 + returns_df['portfolio_return']).cumprod()
        peak = cumulative_returns.cummax()
        drawdown = (cumulative_returns - peak) / peak
        max_drawdown = drawdown.min()
        model = sm.OLS(returns_df['portfolio_return'], sm.add_constant(returns_df['index_return'])).fit()
        beta = model.params[1]
        logger.info(f"Sharpe Ratio: {sharpe_ratio:.4f}, Max Drawdown: {max_drawdown:.4f}, Beta: {beta:.4f}")
    else:
        sharpe_ratio, max_drawdown, beta = np.nan, np.nan, np.nan
    return returns_df, sharpe_ratio, max_drawdown, beta

def main():
    # Define date range
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=365*8)).strftime('%Y-%m-%d')  # ~13 years
    
    # Fetch and preprocess data
    logger.info("Starting data fetch")
    data = fetch_data(NIFTY50, start_date, end_date)
    data = preprocess_data(data)
    index_data = fetch_data(['^NSEI'], start_date, end_date).iloc[:, 0]
    
    # Split data
    train_data, test_data = split_data(data)
    
    # Smooth data
    logger.info("Applying Kalman filter")
    smoothed_data = data.apply(lambda x: apply_kalman_filter(x.dropna()), axis=0).dropna(axis=1, how='any')
    
    # Calculate volatility
    volatility = calculate_volatility(data)
    
    # Rolling regression
    logger.info("Performing rolling regression")
    slopes, p_values, t_stats = {}, {}, {}
    for stock in smoothed_data.columns:
        s, p, t = rolling_regression(smoothed_data[stock])
        slopes[stock] = s
        p_values[stock] = p
        t_stats[stock] = t
    slopes_df = pd.DataFrame(slopes)
    p_values_df = pd.DataFrame(p_values)
    t_stats_df = pd.DataFrame(t_stats)
    
    # Weekly selection dates (last trading day of each week)
    weekly_last_days = data.groupby(pd.Grouper(freq='W')).apply(lambda x: x.index[-1] if not x.empty else pd.NaT).dropna()
    test_start = test_data.index[0]
    selection_dates = [d for d in weekly_last_days if d >= test_start and d in slopes_df.index]
    
    # Select stocks
    logger.info("Selecting momentum stocks")
    selected_stocks = {}
    for date in selection_dates:
        selected_stocks[date] = select_momentum_stocks(slopes_df, p_values_df, t_stats_df, volatility, date)
    
    # Evaluate performance
    logger.info("Evaluating performance")
    returns_df, sharpe_ratio, max_drawdown, beta = evaluate_performance(data, index_data, selected_stocks, selection_dates)
    
    # Log and display results
    if not returns_df.empty:
        logger.info(f"Strategy cumulative return: {returns_df['strategy_cum_return'].iloc[-1]:.4f}")
        logger.info(f"Index cumulative return: {returns_df['index_cum_return'].iloc[-1]:.4f}")
        print(returns_df[['date', 'stocks', 'portfolio_return', 'index_return']].tail())
        print(f"Sharpe Ratio: {sharpe_ratio:.4f}, Max Drawdown: {max_drawdown:.4f}, Beta: {beta:.4f}")
    else:
        logger.warning("No performance data generated")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        logger.critical(f"Program failed: {e}")
        raise